# Model Comparison and Evaluation
5 models will be trained and evaluated on the 4 datasets. The models are:
1. Multi-Head Self-Attention
2. Attention-based BiLSTM
3. Attention-based CNN
4. Hierarchical Attention-based CNN
5. XGBoost with Attention-based Features

Feature Extracted Models:
1. ConvNeXt
2. MobileNetV3Large
3. NASNetLarge
4. Hybrid (ConvNeXt + NASNetLarge)

In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, precision_score, f1_score, matthews_corrcoef, accuracy_score, recall_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb
import time
import os
import warnings
import gc
from tqdm import tqdm
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast

In [47]:
warnings.filterwarnings('ignore')

# Set font family globally
plt.rcParams['font.family'] = 'Times New Roman'

# Base directory
BASE_DIR = "MobileNetV3Large"
os.makedirs(BASE_DIR, exist_ok=True)

device = torch.device("cpu")
try:
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("MPS device detected, using MPS")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
        print("CUDA device detected, using CUDA")
    print(f"Using device: {device}")
except:
    print("Error detecting device capabilities, defaulting to CPU")

In [ ]:
try:
    print("Loading and preparing data...")
    data = pd.read_csv('../3_Feature_Normalization/MobileNetV3Large/files/standardized_data_robust.csv')
    X = data.iloc[:, :2048].values
    y = data['label'].values  # Labels
    num_classes = len(np.unique(y))
    print(f"Dataset size: {X.shape[0]} samples, {X.shape[1]} features, {num_classes} classes")

    # Check class distribution
    print("Class distribution:")
    print(pd.Series(y).value_counts(normalize=True))
except Exception as e:
    print(f"Error loading data: {e}")
    raise

In [ ]:
# Split data
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [ ]:
def create_data_loaders():
    # Convert to PyTorch tensors and reshape
    X_train_3d = torch.FloatTensor(X_train).unsqueeze(-1)
    X_val_3d = torch.FloatTensor(X_val).unsqueeze(-1)
    X_test_3d = torch.FloatTensor(X_test).unsqueeze(-1)
    y_train_t = torch.LongTensor(y_train)
    y_val_t = torch.LongTensor(y_val)
    y_test_t = torch.LongTensor(y_test)

    # DataLoader
    batch_size = 128
    train_dataset = TensorDataset(X_train_3d, y_train_t)
    val_dataset = TensorDataset(X_val_3d, y_val_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    return train_loader, val_loader, X_test_3d, y_test_t

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        e = torch.tanh(self.attention(x))
        alpha = torch.softmax(e, dim=1)
        context = x * alpha
        return context.sum(dim=1)

In [ ]:
class AttBiLSTM(nn.Module):
    def __init__(self, num_classes):
        super(AttBiLSTM, self).__init__()
        self.lstm = nn.LSTM(1, 32, bidirectional=True, batch_first=True)
        self.attention = Attention(64)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x, _ = self.lstm(x)  # [batch, 2048, 64]
        x = self.attention(x)  # [batch, 64]
        x = self.fc(x)  # [batch, num_classes]
        return x

In [ ]:
class AttCNN(nn.Module):
    def __init__(self, num_classes):
        super(AttCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.attention = Attention(32)
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        x = x.transpose(1, 2)  # [batch, 1, 2048]
        x = torch.relu(self.conv1(x))  # [batch, 32, 2048]
        x = self.pool(x)  # [batch, 32, 1024]
        x = x.transpose(1, 2)  # [batch, 1024, 32]
        x = self.attention(x)  # [batch, 32]
        x = self.fc(x)  # [batch, num_classes]
        return x

In [ ]:
class HierarchicalAttCNN(nn.Module):
    def __init__(self, num_classes):
        super(HierarchicalAttCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(32, 16, kernel_size=3, padding=1)
        self.attention = Attention(16)
        self.fc = nn.Linear(16, num_classes)

    def forward(self, x):
        x = x.transpose(1, 2)  # [batch, 1, 2048]
        x = torch.relu(self.conv1(x))  # [batch, 32, 2048]
        x = self.pool1(x)  # [batch, 32, 1024]
        x = torch.relu(self.conv2(x))  # [batch, 16, 1024]
        x = x.transpose(1, 2)  # [batch, 1024, 16]
        x = self.attention(x)  # [batch, 16]
        x = self.fc(x)  # [batch, num_classes]
        return x

In [ ]:
class MultiHeadSelfAtt(nn.Module):
    def __init__(self, num_classes):
        super(MultiHeadSelfAtt, self).__init__()
        self.attn = nn.MultiheadAttention(embed_dim=1, num_heads=1, batch_first=True)
        self.fc = nn.Linear(2048, num_classes)

    def forward(self, x):
        x, _ = self.attn(x, x, x)  # [batch, 2048, 1]
        x = x.reshape(x.size(0), -1)  # [batch, 2048]
        x = self.fc(x)  # [batch, num_classes]
        return x

In [ ]:
def build_att_xgboost(X_train, y_train, X_val, y_val, num_classes):
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    params = {
        'objective': 'multi:softprob' if num_classes > 2 else 'binary:logistic',
        'eval_metric': 'mlogloss' if num_classes > 2 else 'logloss',
        'max_depth': 4,
        'eta': 0.3,
        'num_class': num_classes if num_classes > 2 else None
    }
    evals = [(dtrain, 'train'), (dval, 'val')]
    model = xgb.train(params, dtrain, num_boost_round=50, evals=evals, early_stopping_rounds=5, verbose_eval=False)
    return model

In [ ]:
def train_model(model, train_loader, val_loader, epochs=50, model_name=""):
    try:
        model.to(device)
        class_weights = torch.tensor([1.0 / count for count in pd.Series(y_train).value_counts().sort_index()],
                                     dtype=torch.float).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)

        # optimizer = optim.Adam(model.parameters(), lr=0.0005 if model_name == "Multi-Head Self-Att" else 0.001)
        optimizer = optim.Adam(model.parameters(), 0.001)

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
        use_mixed_precision = device.type == "cuda"
        scaler = GradScaler() if use_mixed_precision else None

        history = {'loss': [], 'val_loss': []}

        for epoch in range(epochs):
            model.train()
            train_loss = 0
            for X_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}"):
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()

                if use_mixed_precision:
                    with autocast():
                        outputs = model(X_batch)
                        loss = criterion(outputs, y_batch)
                    scaler.scale(loss).backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    outputs = model(X_batch)
                    loss = criterion(outputs, y_batch)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

                train_loss += loss.item()
            history['loss'].append(train_loss / len(train_loader))

            model.eval()
            val_loss = 0
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                    outputs = model(X_batch)
                    val_loss += criterion(outputs, y_batch).item()
            val_loss = val_loss / len(val_loader)
            history['val_loss'].append(val_loss)
            scheduler.step(val_loss)

            print(f"Epoch {epoch + 1}/{epochs} - Train Loss: {history['loss'][-1]:.4f}, Val Loss: {val_loss:.4f}")

        return model, history
    except Exception as e:
        print(f"Error in training: {e}")
        raise

In [ ]:
def evaluate_model(model, X_test_3d, y_test, name):
    """Evaluate a PyTorch model and return metrics"""
    model.eval()
    start_time = time.time()
    with torch.no_grad():
        y_pred = model(X_test_3d.to(device)).cpu().numpy()
    test_time = time.time() - start_time
    y_pred_prob = torch.softmax(torch.tensor(y_pred), dim=1).numpy()
    y_pred_binary = np.argmax(y_pred_prob, axis=1)

    results = {
        'ACC': accuracy_score(y_test, y_pred_binary),
        'PRE': precision_score(y_test, y_pred_binary, average='macro'),
        'SP': recall_score(y_test, y_pred_binary, average='macro'),
        'SN': recall_score(y_test, y_pred_binary, average='macro'),
        'F1': f1_score(y_test, y_pred_binary, average='macro'),
        'MCC': matthews_corrcoef(y_test, y_pred_binary),
        'Testing Time': test_time
    }

    # AUC for multiclass (OvR)
    auc_scores = []
    roc_curves = []
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_test == i, y_pred_prob[:, i])
        auc_scores.append(auc(fpr, tpr))
        roc_curves.append((fpr, tpr, auc_scores[-1]))

    results['AUC'] = np.mean(auc_scores)

    # Return metrics and ROC curve data
    return results, roc_curves

In [ ]:
def train_evaluate_xgboost():
    """Train and evaluate XGBoost model"""
    print("\n------------------------------------------------")
    print("Training Att-XGBoost Model")
    print("------------------------------------------------")

    start_time = time.time()
    xgboost_model = build_att_xgboost(X_train, y_train, X_val, y_val, num_classes)
    train_time = time.time() - start_time

    dtest = xgb.DMatrix(X_test)
    start_time = time.time()
    y_pred_prob = xgboost_model.predict(dtest)
    test_time = time.time() - start_time

    y_pred_binary = np.argmax(y_pred_prob, axis=1) if num_classes > 2 else (y_pred_prob > 0.5).astype(int)

    results = {
        'ACC': accuracy_score(y_test, y_pred_binary),
        'PRE': precision_score(y_test, y_pred_binary, average='macro'),
        'SP': recall_score(y_test, y_pred_binary, average='macro'),
        'SN': recall_score(y_test, y_pred_binary, average='macro'),
        'F1': f1_score(y_test, y_pred_binary, average='macro'),
        'MCC': matthews_corrcoef(y_test, y_pred_binary),
        'Training Time': train_time,
        'Testing Time': test_time
    }

    auc_scores = []
    roc_curves = []
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_test == i, y_pred_prob[:, i] if num_classes > 2 else y_pred_prob)
        auc_scores.append(auc(fpr, tpr))
        roc_curves.append((fpr, tpr, auc_scores[-1]))

    results['AUC'] = np.mean(auc_scores)

    # Save XGBoost model
    model_path = os.path.join(BASE_DIR, 'xgboost_model.json')
    xgboost_model.save_model(model_path)
    print(f"XGBoost model saved to {model_path}")

    # Display and save results
    print("\nAtt-XGBoost Results:")
    for metric, value in results.items():
        print(f"{metric}: {value:.4f}")

    # Save results to CSV
    results_df = pd.DataFrame([results], index=['Att-XGBoost'])
    results_df = results_df.round(4)
    results_df.to_csv(os.path.join(BASE_DIR, 'xgboost_metrics.csv'))
    print(f"Results saved to {os.path.join(BASE_DIR, 'xgboost_metrics.csv')}")

    # Return results and ROC data
    return {'Att-XGBoost': results}, {'Att-XGBoost': roc_curves}

In [ ]:
def plot_loss_curves(history, model_name):
    plt.figure(figsize=(10, 6))
    plt.plot(history['loss'], color='blue', linewidth=2, label='Training Loss')
    plt.plot(history['val_loss'], color='red', linewidth=2, label='Validation Loss')
    
    # Set plot styling
    plt.title(f'{model_name} Loss Curves', fontsize=20)
    plt.xlabel('Epochs', fontsize=16)
    plt.ylabel('Loss', fontsize=16)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=14)
    
    # Enhance plot aesthetics
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        spine.set_color('black')
    
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    
    # Save the plot with both PNG and PDF formats
    safe_name = model_name.replace(' ', '_').lower()
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, f'{safe_name}_loss_curves.png'), dpi=1000)
    plt.savefig(os.path.join(BASE_DIR, f'{safe_name}_loss_curves.pdf'))
    plt.close()

In [ ]:
def run_pytorch_model(model_name, model_class):
    print(f"\nTraining {model_name} Model")
    try:
        X_train_subset, X_val, y_train_subset, y_val = train_test_split(X_train, y_train, test_size=0.25,
                                                                        random_state=101)
        train_loader, val_loader, X_test_3d, y_test_t = create_data_loaders(X_train_subset, y_train_subset,
                                                                            X_val, y_val, batch_size=64)

        model = model_class(num_classes)
        start_time = time.time()
        model, history = train_model(model, train_loader, val_loader, epochs=10, model_name=model_name)
        train_time = time.time() - start_time
        
        # NEW: Plot and save the loss curves
        plot_loss_curves(history, model_name)

        results, roc_curves = evaluate_model(model, X_test_3d, y_test, model_name)
        results['Training Time'] = train_time

        # Save and cleanup
        model_path = os.path.join(BASE_DIR, f"{model_name.replace(' ', '_').lower()}.pt")
        torch.save(model.state_dict(), model_path)
        history_df = pd.DataFrame(history)
        history_df.to_csv(os.path.join(BASE_DIR, f"{model_name.replace(' ', '_').lower()}_history.csv"))

        del model, train_loader, val_loader, X_test_3d, y_test_t
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()  # MPS-specific cleanup
        gc.collect()

        print(f"\n{model_name} Results:")
        for metric, value in results.items():
            print(f"{metric}: {value:.4f}")

        return {model_name: results}, {model_name: roc_curves}
    except Exception as e:
        print(f"Error in running {model_name}: {e}")
        return {model_name: {'Error': str(e)}}, {model_name: []}

In [ ]:
def plot_combined_loss_curves():
    """
    Create a combined plot showing the validation loss curves for all models
    """
    plt.figure(figsize=(12, 8))
    
    colors = ['blue', 'green', 'red', 'purple', 'orange']
    models = ['Multi-Head Self-Att', 'Att-BiLSTM', 'Att-CNN', 'Hierarchical Att-CNN']
    
    for i, model_name in enumerate(models):
        try:
            # Load history from CSV
            safe_name = model_name.replace(' ', '_').lower()
            history_path = os.path.join(BASE_DIR, f"{safe_name}_history.csv")
            
            if os.path.exists(history_path):
                history_df = pd.read_csv(history_path)
                plt.plot(history_df['val_loss'], 
                         color=colors[i % len(colors)], 
                         linewidth=2, 
                         label=f'{model_name} Val Loss')
        except Exception as e:
            print(f"Error loading history for {model_name}: {e}")
    
    # Set plot styling
    plt.title('Validation Loss Comparison Across Models', fontsize=20)
    plt.xlabel('Epochs', fontsize=16)
    plt.ylabel('Validation Loss', fontsize=16)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=14)
    
    # Enhance plot aesthetics
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        spine.set_color('black')
    
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, 'combined_val_loss_comparison.png'), dpi=1000)
    plt.savefig(os.path.join(BASE_DIR, 'combined_val_loss_comparison.pdf'), dpi=1000)
    plt.close()

In [ ]:
all_results = {}
all_roc_data = {}

# Run models one by one
for model_name, model_class in [
    ('Multi-Head Self-Att', MultiHeadSelfAtt),
    ('Att-BiLSTM', AttBiLSTM),
    ('Att-CNN', AttCNN),
    ('Hierarchical Att-CNN', HierarchicalAttCNN),
]:
    try:
        results, roc_data = run_pytorch_model(model_name, model_class)
        all_results.update(results)
        all_roc_data.update(roc_data)
    except Exception as e:
        print(f"Error running {model_name}: {e}")
        all_results[model_name] = {'Error': str(e)}

# Split for XGBoost
try:
    X_train_subset, X_val, y_train_subset, y_val = train_test_split(X_train, y_train, test_size=0.25,
                                                                    random_state=101)
    xgboost_results, xgboost_roc = train_evaluate_xgboost(X_train_subset, y_train_subset, X_val, y_val)
    all_results.update(xgboost_results)
    all_roc_data.update(xgboost_roc)
except Exception as e:
    print(f"Error running XGBoost: {e}")
    all_results['Att-XGBoost'] = {'Error': str(e)}

# Create ROC curve plots with thicker lines
plt.figure(figsize=(10, 8))
plt.gca().set_facecolor('white')
colors = ['blue', 'green', 'red', 'purple', 'orange']
for i, (model_name, roc_data) in enumerate(all_roc_data.items()):
    if roc_data:
        for j, (fpr, tpr, auc_value) in enumerate(roc_data):
            if j == 0:
                plt.plot(fpr, tpr, color=colors[i % len(colors)],
                         label=f'{model_name} (AUC = {auc_value:.4f})', linewidth=3)
            else:
                plt.plot(fpr, tpr, color=colors[i % len(colors)], alpha=0, linewidth=1.5)

plt.plot([0, 1], [0, 1], 'k--', linewidth=2)

# Set thickness for all spines
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(2)  # Adjust thickness as needed
    spine.set_color('black')  # Set border color

# Optional: Add a border around the entire plot area
ax.patch.set_linewidth(2)
ax.patch.set_edgecolor('black')

plt.xlabel('False Positive Rate', fontsize=24)
plt.ylabel('True Positive Rate', fontsize=24)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.title('ROC Curves for All Models', fontsize=28, pad=20)
plt.legend(loc='lower right', fontsize=18)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'all_models_roc.png'), dpi=1000)
plt.savefig(os.path.join(BASE_DIR, 'all_models_roc.pdf'), dpi=1000)
plt.show()
plt.close()

# Summary
print("\nSummary of All Models:")
summary_df = pd.DataFrame(all_results).T
column_order = ['ACC', 'AUC', 'PRE', 'SP', 'SN', 'F1', 'MCC', 'Training Time', 'Testing Time']
summary_df = summary_df.reindex(columns=column_order)
summary_df.to_csv(os.path.join(BASE_DIR, 'all_models_summary.csv'))
print(summary_df.round(4))

print("\nGenerating combined validation loss curves...")
plot_combined_loss_curves()
print("Done! All loss curves have been saved to the output directory.")

# Final memory cleanup
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

In [ ]:
print("All done!")